In [1]:
# Import libraries
import os
from os import listdir
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import shapely
import geopandas as gpd
from pyproj import CRS
import utm

In [3]:
# Convert df to gdf
# https://stackoverflow.com/questions/71907567/valueerror-geodataframe-does-not-support-multiple-columns-using-the-geometry-co
# Coordinate system of osm: http://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
# must have a geometry column in csv
def csv_2_gdf(df):
    gdf = gpd.GeoDataFrame(df.loc[:, [c for c in df.columns if c != "geometry"]], 
                           geometry=gpd.GeoSeries.from_wkt(df["geometry"]),
                           crs="epsg:4326",)
    return gdf

### California

In [ ]:
file_path = r"E:\From TRB Folder\OSM_from_Miguel\OSM Database\California"
os.chdir(file_path)

# Reading and saving the links data as a dict
links = {}
# iterate through all file
for file in os.listdir():
    # Check whether file is in text format or not
    if file.endswith("link.csv"):
        # print(file)
        links[file[:-4]] = pd.read_csv(file)  # encoding='cp1252')

print(len(links.keys()))
link_df = pd.concat(links.values()).reset_index(drop=True)
print(link_df.shape) # link_df is a dataframe
print(type(link_df))

In [ ]:
# read the shapefile for that state
gdf = gpd.read_file(r'D:\Work\Box Sync\TRB 2024\US_places_2020\separateExtract\tl_2020_06_place.zip')
len(gdf), type(gdf)

In [ ]:
# assign utm based on the lat long value of a geometry
def utm_crs_from_latlon(lat, lon):
    crs_params = dict(
        proj = 'utm',
        zone = utm.latlon_to_zone_number(lat, lon),
        south = lat < 0
        )
    return CRS.from_dict(crs_params)

# Get projected geometry to create buffer for place polygons
def get_utm_projected(geom_wkt): # geom is shapely geometry file
    geom_shape = shapely.wkt.loads(geom_wkt)
    utm_crs = utm_crs_from_latlon(lat = geom_shape.centroid.y, lon = geom_shape.centroid.x)
    # print(utm_crs)
    line_geo = gpd.GeoSeries([geom_shape], crs='EPSG:4326')
    # taking a buffer of 3.6 meters to include th roads along the periphery
    return line_geo.to_crs(utm_crs).buffer(3.6)

In [ ]:
# clipping roads at state level to places 
link_df_major = link_df[(link_df['link_type'] < 7) | (link_df['link_type'] == 15)]
state_streets_major = csv_2_gdf(link_df_major) 

# Create a new column with sorted tuples
state_streets_major['sorted_nodes'] = list(zip(state_streets_major['from_node_id'], state_streets_major['to_node_id']))
state_streets_major['sorted_nodes'] = state_streets_major['sorted_nodes'].apply(lambda x: tuple(sorted(x)))
   
clipped_to_places = []

state_places = gdf
# cretae a list of names and geoids
geoids = state_places.GEOID
name_of_places = state_places.NAMELSAD
gdf_wkt = state_places.geometry.to_wkt()  

# Clip link gdf by multipolygons
# Creates a list of clipped files per place in states_places
clipped_streets = [gpd.clip(state_streets_major, get_utm_projected(geom_wkt).to_crs(state_streets_major.crs)) for geom_wkt in gdf_wkt]

print(len(clipped_streets))
print(type(clipped_streets))


In [ ]:
clipped_projected_lengths = []
no_roads = []
name_no_roads = []

for i, clipped in enumerate(clipped_streets):
    output_df = pd.DataFrame(clipped)
    # print(len(clipped))    
    print(f'Shape of the roadway clipped by place at index {i} is {clipped.shape}')

    if clipped.shape[0] != 0:
        output_df = pd.DataFrame(clipped)
        utm_length = []
        crs_length = []

        # convert geodataframe geometry to wkt format
        clipped_wkt = clipped.geometry.to_wkt()
        for geom in clipped_wkt:
            # # Convert to accurate UTM: https://gis.stackexchange.com/questions/436938/calculate-length-using-geopandas
            # print(geom)
            # read geometry as a shapely geometry
            line_shape = shapely.wkt.loads(geom)
            # indentify the centroid of the geometry
            utm_crs = utm_crs_from_latlon(lat = line_shape.centroid.y,
                                          lon = line_shape.centroid.x)       

            line_geo = gpd.GeoSeries([line_shape], crs='EPSG:4326')
            crs_length.append(line_geo.to_crs('EPSG:3857').length.values)
            utm_length.append(line_geo.to_crs(utm_crs).length.values)

        if output_df.shape[0] == len(utm_length):
            try:
                output_df['utm_length'] = utm_length
                output_df['crs_length'] = crs_length
            except Exception as e:        
                print(f"!!!====== ERROR ======!!! :\n  {file}")
                print(f"!!!File shapes do not match!!\n")
                continue

        clipped_projected_lengths.append(output_df)
    else:
        print('bleh')
        print('Places and geoids with missing geometry, so no x y values:====')
        print(geoids[i], name_of_places[i])
        no_roads.append(geoids[i])
        name_no_roads.append(name_of_places[i])
        # remove element at index i from the geoid list
        geoids.pop(i)
        name_of_places.pop(i)
        

In [ ]:
print(len(clipped_projected_lengths))

error_opening = []

motorway_roads = []
trunk_roads = []
primary_roads = []
secondary_roads = []
tertiary_roads = []
unclassified_roads = []
residential_roads = []

cl_motorway_roads = []
cl_trunk_roads = []
cl_primary_roads = []
cl_secondary_roads = []
cl_tertiary_roads = []
cl_unclassified_roads = []
cl_residential_roads = []

for k, clipped_links in enumerate(clipped_projected_lengths):
    try:                
        # Filter the data and calculate the sum of road lengths for each road type               
        road_lengths = clipped_links.groupby('link_type_name')['utm_length'].sum()
        cl_road_lengths = clipped_links.groupby(['link_type_name', 'sorted_nodes']).agg({'utm_length':'mean'}).groupby('link_type_name').sum()
        cl_road_lengths = cl_road_lengths.squeeze(axis=1)
        
        # Append the calculated road lengths to the respective lists
        motorway_roads.append(road_lengths.get('motorway', 0))
        trunk_roads.append(road_lengths.get('trunk', 0))
        primary_roads.append(road_lengths.get('primary', 0))
        secondary_roads.append(road_lengths.get('secondary', 0))
        tertiary_roads.append(road_lengths.get('tertiary', 0))
        unclassified_roads.append(road_lengths.get('unclassified', 0))
        residential_roads.append(road_lengths.get('residential', 0))

        cl_motorway_roads.append(cl_road_lengths.get('motorway', 0))
        cl_trunk_roads.append(cl_road_lengths.get('trunk', 0))
        cl_primary_roads.append(cl_road_lengths.get('primary', 0))
        cl_secondary_roads.append(cl_road_lengths.get('secondary', 0))
        cl_tertiary_roads.append(cl_road_lengths.get('tertiary', 0))
        cl_unclassified_roads.append(cl_road_lengths.get('unclassified', 0))
        cl_residential_roads.append(cl_road_lengths.get('residential', 0))
    
            
    except Exception as e:
        error_opening.append(clipped_links.index)
        print(f"Error opening file clipped_links at position {k} of clipped_projected_lengths: {e}")

In [ ]:
list_of_tuples = list(zip(geoids, name_of_places, motorway_roads, trunk_roads, primary_roads, secondary_roads, 
                        tertiary_roads, unclassified_roads, residential_roads, cl_motorway_roads, cl_trunk_roads, cl_primary_roads,
                        cl_secondary_roads, cl_tertiary_roads, cl_unclassified_roads, cl_residential_roads))

df_roads = pd.DataFrame(list_of_tuples, columns = ['GEOID', 'NAMELSAD','motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
                                                'cl_motorway', 'cl_trunk', 'cl_primary', 'cl_secondary', 'cl_tertiary', 'cl_unclassified', 'cl_residential'])

print(df_roads.shape)

df_roads.to_csv(r'D:\Work\Box Sync\Quantify Infrastructure\Streets_df\California\df_roads_utm_california.csv')

missing_places = {no_roads[i]: name_no_roads[i] for i in range(len(no_roads))}
print(len(missing_places))
missing_places

In [ ]:
# ## SAMPLE TRIAL
# clipped_links = clipped_projected_lengths[1]

# road_lengths = clipped_links.groupby('link_type_name')['utm_length'].sum()
# cl_road_lengths = clipped_links.groupby(['link_type_name', 'sorted_nodes']).agg({'utm_length':'mean'}).groupby('link_type_name').sum()
# # cl_road_lengths = cl_road_lengths.squeeze(axis=1)
# print(type(cl_road_lengths))  
# cl_road_lengths.squeeze()

### Florida

In [ ]:
file_path = r"E:\From TRB Folder\OSM_from_Miguel\OSM Database\Florida"
os.chdir(file_path)

# Reading and saving the links data as a dict
links = {}
# iterate through all file
for file in os.listdir():
    # Check whether file is in text format or not
    if file.endswith("link.csv"):
        # print(file)
        links[file[:-4]] = pd.read_csv(file)  # encoding='cp1252')

print(len(links.keys()))
link_df = pd.concat(links.values()).reset_index(drop=True)
print(link_df.shape) # link_df is a dataframe
print(type(link_df))

In [ ]:
# read the shapefile for that state
gdf = gpd.read_file(r'D:\Work\Box Sync\TRB 2024\US_places_2020\separateExtract\tl_2020_12_place.zip')
len(gdf), type(gdf)

In [ ]:
# assign utm based on the lat long value of a geometry
def utm_crs_from_latlon(lat, lon):
    crs_params = dict(
        proj = 'utm',
        zone = utm.latlon_to_zone_number(lat, lon),
        south = lat < 0
        )
    return CRS.from_dict(crs_params)

# Get projected geometry to create buffer for place polygons
def get_utm_projected(geom_wkt): # geom is shapely geometry file
    geom_shape = shapely.wkt.loads(geom_wkt)
    utm_crs = utm_crs_from_latlon(lat = geom_shape.centroid.y, lon = geom_shape.centroid.x)
    # print(utm_crs)
    line_geo = gpd.GeoSeries([geom_shape], crs='EPSG:4326')
    # taking a buffer of 3.6 meters to include th roads along the periphery
    return line_geo.to_crs(utm_crs).buffer(3.6)

In [ ]:
# clipping roads at state level to places 
link_df_major = link_df[(link_df['link_type'] < 7) | (link_df['link_type'] == 15)]
state_streets_major = csv_2_gdf(link_df_major) 

# Create a new column with sorted tuples
state_streets_major['sorted_nodes'] = list(zip(state_streets_major['from_node_id'], state_streets_major['to_node_id']))
state_streets_major['sorted_nodes'] = state_streets_major['sorted_nodes'].apply(lambda x: tuple(sorted(x)))
   
clipped_to_places = []

state_places = gdf
# cretae a list of names and geoids
geoids = state_places.GEOID
name_of_places = state_places.NAMELSAD
gdf_wkt = state_places.geometry.to_wkt()  

# Clip link gdf by multipolygons
# Creates a list of clipped files per place in states_places
clipped_streets = [gpd.clip(state_streets_major, get_utm_projected(geom_wkt).to_crs(state_streets_major.crs)) for geom_wkt in gdf_wkt]

print(len(clipped_streets))
print(type(clipped_streets))


In [ ]:
clipped_projected_lengths = []
no_roads = []
name_no_roads = []

for i, clipped in enumerate(clipped_streets):
    # print(len(clipped))    
    clipped = clipped[~clipped.is_empty]
    print(f'Shape of the roadway clipped by place at index {i} is {clipped.shape}')
    if clipped.shape[0] != 0:
        output_df = pd.DataFrame(clipped)
        utm_length = []
        crs_length = []

        # convert geodataframe geometry to wkt format
        clipped_wkt = clipped.geometry.to_wkt()
        for geom in clipped_wkt:
            # # Convert to accurate UTM: https://gis.stackexchange.com/questions/436938/calculate-length-using-geopandas
            # print(geom)
            # read geometry as a shapely geometry
            line_shape = shapely.wkt.loads(geom)
            # indentify the centroid of the geometry
            utm_crs = utm_crs_from_latlon(lat = line_shape.centroid.y,
                                          lon = line_shape.centroid.x)       

            line_geo = gpd.GeoSeries([line_shape], crs='EPSG:4326')
            crs_length.append(line_geo.to_crs('EPSG:3857').length.values)
            utm_length.append(line_geo.to_crs(utm_crs).length.values)

        if output_df.shape[0] == len(utm_length):
            try:
                output_df['utm_length'] = utm_length
                output_df['crs_length'] = crs_length
            except Exception as e:        
                print(f"!!!====== ERROR ======!!! :\n  {file}")
                print(f"!!!File shapes do not match!!\n")
                continue

        clipped_projected_lengths.append(output_df)
    else:
        print('bleh')
        print('Places and geoids with missing geometry, so no x y values:====')
        print(geoids[i], name_of_places[i])
        no_roads.append(geoids[i])
        name_no_roads.append(name_of_places[i])
        # remove element at index i from the geoid list
        geoids.pop(i)
        name_of_places.pop(i)
        

In [ ]:
print(len(clipped_projected_lengths))

error_opening = []

motorway_roads = []
trunk_roads = []
primary_roads = []
secondary_roads = []
tertiary_roads = []
unclassified_roads = []
residential_roads = []

cl_motorway_roads = []
cl_trunk_roads = []
cl_primary_roads = []
cl_secondary_roads = []
cl_tertiary_roads = []
cl_unclassified_roads = []
cl_residential_roads = []

for k, clipped_links in enumerate(clipped_projected_lengths):
    try:                
        # Filter the data and calculate the sum of road lengths for each road type               
        road_lengths = clipped_links.groupby('link_type_name')['utm_length'].sum()
        cl_road_lengths = clipped_links.groupby(['link_type_name', 'sorted_nodes']).agg({'utm_length':'mean'}).groupby('link_type_name').sum()
        cl_road_lengths = cl_road_lengths.squeeze(axis=1)
        
        # Append the calculated road lengths to the respective lists
        motorway_roads.append(road_lengths.get('motorway', 0))
        trunk_roads.append(road_lengths.get('trunk', 0))
        primary_roads.append(road_lengths.get('primary', 0))
        secondary_roads.append(road_lengths.get('secondary', 0))
        tertiary_roads.append(road_lengths.get('tertiary', 0))
        unclassified_roads.append(road_lengths.get('unclassified', 0))
        residential_roads.append(road_lengths.get('residential', 0))

        cl_motorway_roads.append(cl_road_lengths.get('motorway', 0))
        cl_trunk_roads.append(cl_road_lengths.get('trunk', 0))
        cl_primary_roads.append(cl_road_lengths.get('primary', 0))
        cl_secondary_roads.append(cl_road_lengths.get('secondary', 0))
        cl_tertiary_roads.append(cl_road_lengths.get('tertiary', 0))
        cl_unclassified_roads.append(cl_road_lengths.get('unclassified', 0))
        cl_residential_roads.append(cl_road_lengths.get('residential', 0))
    
            
    except Exception as e:
        error_opening.append(clipped_links.index)
        print(f"Error opening file clipped_links at position {k} of clipped_projected_lengths: {e}")

In [ ]:
list_of_tuples = list(zip(geoids, name_of_places, motorway_roads, trunk_roads, primary_roads, secondary_roads, 
                        tertiary_roads, unclassified_roads, residential_roads, cl_motorway_roads, cl_trunk_roads, cl_primary_roads,
                        cl_secondary_roads, cl_tertiary_roads, cl_unclassified_roads, cl_residential_roads))

df_roads = pd.DataFrame(list_of_tuples, columns = ['GEOID', 'NAMELSAD','motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
                                                'cl_motorway', 'cl_trunk', 'cl_primary', 'cl_secondary', 'cl_tertiary', 'cl_unclassified', 'cl_residential'])

print(df_roads.shape)

print(df_roads.shape)
df_roads.to_csv(r'D:\Work\Box Sync\Quantify Infrastructure\Streets_df\Florida\df_roads_utm_florida_'+ str(i) + '.csv')

missing_places = {no_roads[i]: name_no_roads[i] for i in range(len(no_roads))}
print(len(missing_places))

In [ ]:
missing_places

### Texas

In [4]:
file_path = r"E:\From TRB Folder\OSM_from_Miguel\OSM Database\Texas"
os.chdir(file_path)

# Reading and saving the links data as a dict
links = {}
# iterate through all file
for file in os.listdir():
    # Check whether file is in text format or not
    if file.endswith("link.csv"):
        # print(file)
        links[file[:-4]] = pd.read_csv(file)  # encoding='cp1252')

print(len(links.keys()))
link_df = pd.concat(links.values()).reset_index(drop=True)
print(link_df.shape) # link_df is a dataframe
print(type(link_df))

254
(12740826, 18)
<class 'pandas.core.frame.DataFrame'>


In [5]:
# read the shapefile for that state
gdf = gpd.read_file(r'D:\Work\Box Sync\TRB 2024\US_places_2020\separateExtract\tl_2020_48_place.zip')
len(gdf), type(gdf)

(1860, geopandas.geodataframe.GeoDataFrame)

In [6]:
# assign utm based on the lat long value of a geometry
def utm_crs_from_latlon(lat, lon):
    crs_params = dict(
        proj = 'utm',
        zone = utm.latlon_to_zone_number(lat, lon),
        south = lat < 0
        )
    return CRS.from_dict(crs_params)

# Get projected geometry to create buffer for place polygons
def get_utm_projected(geom_wkt): # geom is shapely geometry file
    geom_shape = shapely.wkt.loads(geom_wkt)
    utm_crs = utm_crs_from_latlon(lat = geom_shape.centroid.y, lon = geom_shape.centroid.x)
    # print(utm_crs)
    line_geo = gpd.GeoSeries([geom_shape], crs='EPSG:4326')
    # taking a buffer of 3.6 meters to include th roads along the periphery
    return line_geo.to_crs(utm_crs).buffer(3.6)

In [7]:
# clipping roads at state level to places 
link_df_major = link_df[(link_df['link_type'] < 7) | (link_df['link_type'] == 15)]
state_streets_major = csv_2_gdf(link_df_major) 

# Create a new column with sorted tuples
state_streets_major['sorted_nodes'] = list(zip(state_streets_major['from_node_id'], state_streets_major['to_node_id']))
state_streets_major['sorted_nodes'] = state_streets_major['sorted_nodes'].apply(lambda x: tuple(sorted(x)))
   
clipped_to_places = []

state_places = gdf
# cretae a list of names and geoids
geoids = state_places.GEOID
name_of_places = state_places.NAMELSAD
gdf_wkt = state_places.geometry.to_wkt()  

# Clip link gdf by multipolygons
# Creates a list of clipped files per place in states_places
clipped_streets = [gpd.clip(state_streets_major, get_utm_projected(geom_wkt).to_crs(state_streets_major.crs)) for geom_wkt in gdf_wkt]

print(len(clipped_streets))
print(type(clipped_streets))


1860
<class 'list'>


In [8]:
clipped_projected_lengths = []
no_roads = []
name_no_roads = []

for i, clipped in enumerate(clipped_streets):
    clipped = clipped[~clipped.is_empty]
    # print(len(clipped))    
    print(f'Shape of the roadway clipped by place at index {i} is {clipped.shape}')

    if clipped.shape[0] != 0:
        output_df = pd.DataFrame(clipped)
        utm_length = []
        crs_length = []

        # convert geodataframe geometry to wkt format
        clipped_wkt = clipped.geometry.to_wkt()
        for geom in clipped_wkt:
            # # Convert to accurate UTM: https://gis.stackexchange.com/questions/436938/calculate-length-using-geopandas
            # print(geom)
            # read geometry as a shapely geometry
            line_shape = shapely.wkt.loads(geom)
            # indentify the centroid of the geometry
            utm_crs = utm_crs_from_latlon(lat = line_shape.centroid.y,
                                          lon = line_shape.centroid.x)       

            line_geo = gpd.GeoSeries([line_shape], crs='EPSG:4326')
            crs_length.append(line_geo.to_crs('EPSG:3857').length.values)
            utm_length.append(line_geo.to_crs(utm_crs).length.values)

        if output_df.shape[0] == len(utm_length):
            try:
                output_df['utm_length'] = utm_length
                output_df['crs_length'] = crs_length
            except Exception as e:        
                print(f"!!!====== ERROR ======!!! :\n  {file}")
                print(f"!!!File shapes do not match!!\n")
                continue

        clipped_projected_lengths.append(output_df)
    else:
        print('bleh')
        print('Places and geoids with missing geometry, so no x y values:====')
        print(geoids[i], name_of_places[i])
        no_roads.append(geoids[i])
        name_no_roads.append(name_of_places[i])
        # remove element at index i from the geoid list
        geoids.pop(i)
        name_of_places.pop(i)
        

Shape of the roadway clipped by place at index 0 is (4045, 19)
Shape of the roadway clipped by place at index 1 is (434, 19)
Shape of the roadway clipped by place at index 2 is (1533, 19)
Shape of the roadway clipped by place at index 3 is (222, 19)
Shape of the roadway clipped by place at index 4 is (262, 19)
Shape of the roadway clipped by place at index 5 is (148, 19)
Shape of the roadway clipped by place at index 6 is (1144, 19)
Shape of the roadway clipped by place at index 7 is (6587, 19)
Shape of the roadway clipped by place at index 8 is (276, 19)
Shape of the roadway clipped by place at index 9 is (640, 19)
Shape of the roadway clipped by place at index 10 is (780, 19)
Shape of the roadway clipped by place at index 11 is (234, 19)
Shape of the roadway clipped by place at index 12 is (166, 19)
Shape of the roadway clipped by place at index 13 is (874, 19)
Shape of the roadway clipped by place at index 14 is (1901, 19)
Shape of the roadway clipped by place at index 15 is (704, 1

In [9]:
print(len(clipped_projected_lengths))

error_opening = []

motorway_roads = []
trunk_roads = []
primary_roads = []
secondary_roads = []
tertiary_roads = []
unclassified_roads = []
residential_roads = []

cl_motorway_roads = []
cl_trunk_roads = []
cl_primary_roads = []
cl_secondary_roads = []
cl_tertiary_roads = []
cl_unclassified_roads = []
cl_residential_roads = []

for k, clipped_links in enumerate(clipped_projected_lengths):
    try:                
        # Filter the data and calculate the sum of road lengths for each road type               
        road_lengths = clipped_links.groupby('link_type_name')['utm_length'].sum()
        cl_road_lengths = clipped_links.groupby(['link_type_name', 'sorted_nodes']).agg({'utm_length':'mean'}).groupby('link_type_name').sum()
        cl_road_lengths = cl_road_lengths.squeeze(axis=1)
        
        # Append the calculated road lengths to the respective lists
        motorway_roads.append(road_lengths.get('motorway', 0))
        trunk_roads.append(road_lengths.get('trunk', 0))
        primary_roads.append(road_lengths.get('primary', 0))
        secondary_roads.append(road_lengths.get('secondary', 0))
        tertiary_roads.append(road_lengths.get('tertiary', 0))
        unclassified_roads.append(road_lengths.get('unclassified', 0))
        residential_roads.append(road_lengths.get('residential', 0))

        cl_motorway_roads.append(cl_road_lengths.get('motorway', 0))
        cl_trunk_roads.append(cl_road_lengths.get('trunk', 0))
        cl_primary_roads.append(cl_road_lengths.get('primary', 0))
        cl_secondary_roads.append(cl_road_lengths.get('secondary', 0))
        cl_tertiary_roads.append(cl_road_lengths.get('tertiary', 0))
        cl_unclassified_roads.append(cl_road_lengths.get('unclassified', 0))
        cl_residential_roads.append(cl_road_lengths.get('residential', 0))
        print("finished upto ", str(k))
    
            
    except Exception as e:
        error_opening.append(clipped_links.index)
        print(f"Error opening file clipped_links at position {k} of clipped_projected_lengths: {e}")

1860
finished upto  0
finished upto  1
finished upto  2
finished upto  3
finished upto  4
finished upto  5
finished upto  6
finished upto  7
finished upto  8
finished upto  9
finished upto  10
finished upto  11
finished upto  12
finished upto  13
finished upto  14
finished upto  15
finished upto  16
finished upto  17
finished upto  18
finished upto  19
finished upto  20
finished upto  21
finished upto  22
finished upto  23
finished upto  24
finished upto  25
finished upto  26
finished upto  27
finished upto  28
finished upto  29
finished upto  30
finished upto  31
finished upto  32
finished upto  33
finished upto  34
finished upto  35
finished upto  36
finished upto  37
finished upto  38
finished upto  39
finished upto  40
finished upto  41
finished upto  42
finished upto  43
finished upto  44
finished upto  45
finished upto  46
finished upto  47
finished upto  48
finished upto  49
finished upto  50
finished upto  51
finished upto  52
finished upto  53
finished upto  54
finished upto  

In [10]:
list_of_tuples = list(zip(geoids, name_of_places, motorway_roads, trunk_roads, primary_roads, secondary_roads, 
                        tertiary_roads, unclassified_roads, residential_roads, cl_motorway_roads, cl_trunk_roads, cl_primary_roads,
                        cl_secondary_roads, cl_tertiary_roads, cl_unclassified_roads, cl_residential_roads))

df_roads = pd.DataFrame(list_of_tuples, columns = ['GEOID', 'NAMELSAD','motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
                                                'cl_motorway', 'cl_trunk', 'cl_primary', 'cl_secondary', 'cl_tertiary', 'cl_unclassified', 'cl_residential'])

print(df_roads.shape)

print(df_roads.shape)
df_roads.to_csv(r'D:\Work\Box Sync\Quantify Infrastructure\Streets_df\Texas\df_roads_utm_texas.csv')

missing_places = {no_roads[i]: name_no_roads[i] for i in range(len(no_roads))}
print(len(missing_places))
missing_places

(1860, 16)
(1860, 16)
0


{}